In [1]:
import string
import torch
import numpy as np
from tqdm import tqdm
import torch.nn as nn
import albumentations as A
import torch.nn.functional as F
import glob , json , timm , cv2 , random
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchsummary import summary
from PIL import Image, ImageDraw, ImageFont

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"device: {device}")

device: cuda


### Datasets

In [2]:
IMG_W = 256
IMG_H = 32

CHARS = (
    string.digits +
    string.ascii_uppercase +
    string.ascii_lowercase + 
    " "
)

char_to_idx = {c: i + 1 for i, c in enumerate(CHARS)}
idx_to_char = {i + 1: c for i, c in enumerate(CHARS)}
BLANK_IDX = 0

# --- Load the Custom Fonts ---
# Checks the root fonts folder and one sub-directory deep
FONT_PATHS = glob.glob("fonts_simple/*/*/*.[to]tf")  + glob.glob("fonts/*/*/*/*.[to]tf") 
if not FONT_PATHS:
    print("WARNING: No fonts found! Please add .ttf or .otf files to a 'fonts' directory.")
## FONT_PATHS = FONT_PATHS[:1]

def get_random_colors():
    """Generates contrasting random colors for background and text."""
    bg = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
    
    # Ensure text contrasts with the background (simulating clean UI elements)
    if sum(bg) > 382: # Light background -> Dark text
        txt = (random.randint(0, 100), random.randint(0, 100), random.randint(0, 100))
    else:             # Dark background -> Light text
        txt = (random.randint(150, 255), random.randint(150, 255), random.randint(150, 255))
    return bg, txt

def generate_simple_synthetic():
    """Generates tightly cropped text scaled to height 32, simulating a detector pipeline."""
    # 1. Generate text and colors
    length = random.randint(3, 12)
    text = "".join(random.choice(CHARS) for _ in range(length))
    bg_color, txt_color = get_random_colors()

    # 2. Setup font
    font_path = random.choice(FONT_PATHS)
    # Start with a large base font to render high-quality pixels before scaling down
    font_size = random.randint(32, 48) 
    font = ImageFont.truetype(font_path, font_size)

    # 3. Get EXACT text bounding box dimensions
    dummy_img = Image.new('RGB', (1, 1))
    dummy_draw = ImageDraw.Draw(dummy_img)
    bbox = dummy_draw.textbbox((0, 0), text, font=font)
    
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]

    # 4. Simulate imperfect detector crops (0 to 3 pixels of random padding)
    pad_x = random.randint(0, 3)
    pad_y = random.randint(0, 3)
    
    crop_w = max(1, tw + (pad_x * 2))
    crop_h = max(1, th + (pad_y * 2))

    # 5. Draw the tight crop
    tight_img = Image.new('RGB', (crop_w, crop_h), color=bg_color)
    draw = ImageDraw.Draw(tight_img)
    # Offset by bbox[0] and bbox[1] because some fonts have negative starting coordinates
    draw.text((pad_x - bbox[0], pad_y - bbox[1]), text, font=font, fill=txt_color)

    # 6. Scale height to exactly 32 (IMG_H) and adjust width to maintain aspect ratio
    scale = IMG_H / crop_h
    new_w = min(int(crop_w * scale), IMG_W) # Cap max width at 256
    
    # Use BILINEAR or LANCZOS for high-quality downsampling
    resized_img = tight_img.resize((new_w, IMG_H), Image.Resampling.BILINEAR)

    # 7. Create the final 256x32 canvas
    final_img = Image.new('RGB', (IMG_W, IMG_H), color=bg_color)
    
    # Left-align the text with a slight jitter (0 to 10 pixels) 
    # CTC models prefer text starting on the left rather than centered
    max_jitter = max(0, IMG_W - new_w)
    x_off = random.randint(0, min(10, max_jitter))
    
    final_img.paste(resized_img, (x_off, 0))

    # 8. Convert to Tensor
    img_np = np.array(final_img).astype(np.float32) / 255.0
    img_tensor = np.transpose(img_np, (2, 0, 1)) 
    
    return img_tensor, text


class SimpleOCRDataset(Dataset):
    def __init__(self, real_data_list=None, epoch_size=100000, real_prob=0.2):
        self.real_data = real_data_list or []
        self.epoch_size = epoch_size
        self.real_prob = real_prob

    def __len__(self):
        return self.epoch_size

    def __getitem__(self, idx):
        if random.random() < self.real_prob and len(self.real_data) > 0:
            img_path, text = random.choice(self.real_data)
            img = cv2.imread(img_path, cv2.IMREAD_COLOR) # Load as BGR color
            
            if img is None:
                img_tensor, text = generate_simple_synthetic()
            else:
                img_tensor = self._preprocess_real(img)
        else:
            img_tensor, text = generate_simple_synthetic()

        return torch.tensor(img_tensor, dtype=torch.float32), text

    def _preprocess_real(self, color_img):
        # Convert BGR (OpenCV) to RGB to match the synthetic PIL generations
        color_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2RGB)
        
        h, w, _ = color_img.shape
        scale = min(IMG_W / w, IMG_H / h)
        new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
        
        resized = cv2.resize(color_img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        
        # Pad with black background for real images
        canvas = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)
        
        x_off, y_off = (IMG_W - new_w) // 2, (IMG_H - new_h) // 2
        canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
        
        canvas = canvas.astype(np.float32) / 255.0
        return np.transpose(canvas, (2, 0, 1))

# --- Keep your existing Helper Functions (collate_fn, ctc_decode, levenshtein) ---

def collate_fn(batch):
    images, texts = zip(*batch)
    images = torch.stack(images)
    
    targets = []
    target_lengths = []
    
    for text in texts:
        encoded = [char_to_idx[c] for c in text if c in char_to_idx]
        targets.extend(encoded)
        target_lengths.append(len(encoded)) 
        
    targets = torch.tensor(targets, dtype=torch.long)
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)
    
    return images, targets, target_lengths, texts

def ctc_decode(logits):
    pred = logits.argmax(-1)
    text = []
    prev = -1
    for p in pred:
        p = p.item()
        if p != prev and p != 0:
            text.append(idx_to_char[p])
        prev = p
    return "".join(text)

def levenshtein_distance(s1, s2):
    if len(s1) > len(s2):
        s1, s2 = s2, s1
    distances = range(len(s1) + 1)
    for i2, c2 in enumerate(s2):
        distances_ = [i2+1]
        for i1, c1 in enumerate(s1):
            if c1 == c2:
                distances_.append(distances[i1])
            else:
                distances_.append(1 + min((distances[i1], distances[i1 + 1], distances_[-1])))
        distances = distances_
    return distances[-1]

### Model Definition

In [3]:
class EnhancedOCRNet(nn.Module):
    def __init__(self, num_chars, dropout_p=0.1):
        super().__init__()
        
        # 1. Full-width small backbone (Boosts CNN feature extraction for punctuation)
        self.backbone = timm.create_model(
            "mobilenetv3_small_100", 
            pretrained=True, 
            features_only=True, 
            out_indices=(2,) 
        )
        
        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, 32, 256)
            out_channels = self.backbone(dummy_input)[0].shape[1] # Outputs 24
        
        # 2. Dual-layer RNN (Recovers sequence memory for long, complex strings)
        self.rnn = nn.GRU(
            input_size=out_channels, 
            hidden_size=96,       
            num_layers=2,         # <-- Restored to 2 layers
            bidirectional=True,
            batch_first=True,
            dropout=dropout_p 
        )
        
        # 3. High-Capacity Head (192 -> 192 to completely eliminate the bottleneck)
        self.head = nn.Sequential(
            nn.Linear(192, 192),   
            nn.Hardswish(inplace=True),
            nn.LayerNorm(192),
            nn.Dropout(dropout_p), 
            nn.Linear(192, num_chars + 1)
        )

    def forward(self, x):
        x = self.backbone(x)[0]
        x = x.mean(dim=2)
        x = x.permute(0, 2, 1)
        x, _ = self.rnn(x)  
        x = self.head(x)  
        return x

In [4]:
from torchinfo import summary
model = EnhancedOCRNet(num_chars=len(CHARS))
model.cpu() 
dummy_input = torch.zeros(1, 3, 32, 256)
summary(model, input_data=dummy_input,col_names=["input_size", "output_size", "num_params", "trainable"],col_width=20,row_settings=["var_names"])

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Layer (type (var_name))                                 Input Shape          Output Shape         Param #              Trainable
EnhancedOCRNet (EnhancedOCRNet)                         [1, 3, 32, 256]      [1, 32, 64]          --                   True
├─MobileNetV3Features (backbone)                        [1, 3, 32, 256]      [1, 24, 4, 32]       --                   True
│    └─Conv2d (conv_stem)                               [1, 3, 32, 256]      [1, 16, 16, 128]     432                  True
│    └─BatchNorm2d (bn1)                                [1, 16, 16, 128]     [1, 16, 16, 128]     32                   True
│    └─Hardswish (act1)                                 [1, 16, 16, 128]     [1, 16, 16, 128]     --                   --
│    └─Sequential (blocks)                              --                   --                   --                   True
│    │    └─Sequential (0)                              [1, 16, 16, 128]     [1, 16, 8, 64]       744                  True
│    

### Model Training

In [5]:
### Loss func
class FocalCTCLoss(nn.Module):
    def __init__(self, blank=0, alpha=1.0, gamma=2.0):
        super().__init__()
        self.ctc = nn.CTCLoss(blank=blank, zero_infinity=True, reduction='none')
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, log_probs, targets, input_lengths, target_lengths):
        loss = self.ctc(log_probs, targets, input_lengths, target_lengths)
        
        # CRITICAL SHIELD: Clamping prevents negative garbage values 
        # from triggering an exponential math explosion.
        loss = torch.clamp(loss, min=0.0)
        
        p = torch.exp(-loss)
        focal_loss = self.alpha * ((1 - p) ** self.gamma) * loss
        
        return focal_loss.mean()




train_dataset = SimpleOCRDataset(epoch_size=102400)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, num_workers=12 , collate_fn=collate_fn, prefetch_factor=2,persistent_workers=True)
val_dataset = SimpleOCRDataset(epoch_size=10240)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False, num_workers=12, collate_fn=collate_fn,prefetch_factor=2,persistent_workers=True)

model = EnhancedOCRNet(num_chars=len(CHARS)).to(device)
ctc_loss = FocalCTCLoss(blank=0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [ ]:
EPOCHS = 100
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_char_acc = 0.0
patience = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    
    for images, targets, target_lengths, _ in pbar:
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()

        logits = model(images)
        logits = logits.permute(1, 0, 2)
        log_probs = F.log_softmax(logits, dim=2)

        T, B = log_probs.size(0), log_probs.size(1)
        input_lengths = torch.full(size=(B,), fill_value=T, dtype=torch.long, device=device)
        
        loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        running_loss += loss.item() 
        pbar.set_postfix(loss=loss.item())

    scheduler.step()
    epoch_loss = running_loss / len(train_loader)
    
    # --- VALIDATION ---
    model.eval()
    total_edit_distance = 0
    total_gt_chars = 0
    
    with torch.no_grad():
        for val_images, _, _, val_texts in val_loader:
            val_images = val_images.to(device)
            val_logits = model(val_images)
            
            for idx in range(val_images.size(0)):
                pred_text = ctc_decode(val_logits[idx])
                gt_text = val_texts[idx]
                
                total_edit_distance += levenshtein_distance(pred_text, gt_text)
                total_gt_chars += max(len(gt_text), 1)
                
    char_accuracy = (1.0 - (total_edit_distance / total_gt_chars)) * 100
    print(f"Epoch {epoch} Summary: Loss={epoch_loss:.4f} | Char Accuracy={char_accuracy:.2f}%")
    
    # --- CHECKPOINTING & EARLY STOPPING ---
    checkpoint = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'best_acc': best_char_acc
    }
    torch.save(checkpoint, "latest_ocr_big.pth")
    
    if char_accuracy > best_char_acc:
        best_char_acc = char_accuracy
        torch.save(checkpoint, "best_ocr_big.pth")
        print(f"Saved Checkpoint with Char Accuracy: {best_char_acc:.2f}%")
        patience = 0
    else:
        patience += 1

    if patience >= 50:
        print(f"Early stopping triggered at epoch {epoch}")
        break

Epoch 0/100: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:27<00:00,  1.14it/s, loss=11.8]


Epoch 0 Summary: Loss=25.5796 | Char Accuracy=45.06%
Saved Checkpoint with Char Accuracy: 45.06%


Epoch 1/100: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:30<00:00,  1.10it/s, loss=4.03]


Epoch 1 Summary: Loss=6.9796 | Char Accuracy=83.20%
Saved Checkpoint with Char Accuracy: 83.20%


Epoch 2/100: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:23<00:00,  1.19it/s, loss=1.86]


...

Epoch 27/100: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [02:40<00:00,  1.60s/it, loss=0.206]


Epoch 27 Summary: Loss=0.2325 | Char Accuracy=98.23%


Epoch 28/100: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [02:37<00:00,  1.57s/it, loss=0.241]


Epoch 28 Summary: Loss=0.2294 | Char Accuracy=98.29%


Epoch 29/100:   0%|                                                                                                                                            | 0/100 [00:00<?, ?it/s]

### Evaluation of model

In [ ]:
model.eval()
print("\n--- Final Model Evaluation ---")
for i in range(10):
    img, gt = generate_sample()
    x = torch.tensor(img).float().unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)[0]
    pred = ctc_decode(logits)
    
    print(f"GT  : {gt}")
    print(f"PRED: {pred}")

In [ ]:

img_tensor, text = generate_simple_synthetic()

# Transpose from (3, 32, 256) to (32, 256, 3)


img_display = np.transpose(img_tensor, (1, 2, 0))

plt.imshow(img_display)
